## final project 

problem statement - prducting house price(califrnia)

> on real world dataset

<img src="https://www.appliedaicourse.com/blog/wp-content/uploads/2025/01/House-Price-Prediction-Using-Machine-Learning.png">



## California House Price Prediction Project

### Objective
> The objective of this project is to build a multiple linear regression model that predicts the median house value in California districts based on various features like median income, housing median age, and average room counts etc...

#### Machine Learning Pipeline

* Data Collection and Import
* Exploratory Data Analysis & Data Cleaning (Handling Nulls and Duplicates)
* Feature Selection & Data Splitting
* Feature Scaling (Standardization)
* Model Building & Training
* Model Evaluation

# Step 1: Data Collection and Import
# Load the dataset from a URL or local file path

>> importing required libraries

In [23]:
import pandas as pd
import numpy as np

In [24]:
data_url = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(data_url)

In [25]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [26]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   longitude           20640 non-null  float64
 1   latitude            20640 non-null  float64
 2   housing_median_age  20640 non-null  float64
 3   total_rooms         20640 non-null  float64
 4   total_bedrooms      20433 non-null  float64
 5   population          20640 non-null  float64
 6   households          20640 non-null  float64
 7   median_income       20640 non-null  float64
 8   median_house_value  20640 non-null  float64
 9   ocean_proximity     20640 non-null  str    
dtypes: float64(9), str(1)
memory usage: 1.6 MB


> in the feature called " total_bedrooms" we have null values so we need fix it (by mean/median)

In [27]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

Duplicate rows: 0


In [28]:
df.shape

(20640, 10)

In [29]:
# Handle missing values: Fill missing 'total_bedrooms' with the median of that column
median_bedrooms=df["total_bedrooms"].median()
df["total_bedrooms"].fillna(median_bedrooms, inplace=True)

C:\Users\klegd\AppData\Local\Temp\ipykernel_15100\1253006522.py:3: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["total_bedrooms"].fillna(median_bedrooms, inplace=True)


0         129.0
1        1106.0
2         190.0
3         235.0
4         280.0
          ...  
20635     374.0
20636     150.0
20637     485.0
20638     409.0
20639     616.0
Name: total_bedrooms, Length: 20640, dtype: float64

In [30]:
df["total_bedrooms"].isnull().sum()

np.int64(207)

In [31]:
### Enhancing the Model: One-Hot Encoding and Feature Engineering

# one-hot encoding to 'ocean_proximity'
df_encoded = pd.get_dummies(df, columns=['ocean_proximity'], drop_first=True)
df_encoded.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,False,False,True,False
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,False,False,True,False
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,False,False,True,False
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,False,False,True,False
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,False,False,True,False


#### feature engineering...

> creating new features (based on other features(column,input,vectors))

In [32]:
# create new features
df_encoded['rooms_per_household'] = df_encoded['total_rooms'] / df_encoded['households']
df_encoded['bedrooms_per_room'] = df_encoded['total_bedrooms'] / df_encoded['total_rooms']
df_encoded['population_per_household'] = df_encoded['population'] / df_encoded['households']

df_encoded.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN,rooms_per_household,bedrooms_per_room,population_per_household
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,False,False,True,False,6.984127,0.146591,2.555556
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,False,False,True,False,6.238137,0.155797,2.109842
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,False,False,True,False,8.288136,0.129516,2.802260
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,False,False,True,False,5.817352,0.184458,2.547945
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,False,False,True,False,6.281853,0.172096,2.181467


### vetical split(i/p and o/p)

In [33]:
X=df_encoded.drop(columns = "median_house_value")#i/p dataset

In [34]:
X.columns

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'ocean_proximity_INLAND', 'ocean_proximity_ISLAND',
       'ocean_proximity_NEAR BAY', 'ocean_proximity_NEAR OCEAN',
       'rooms_per_household', 'bedrooms_per_room', 'population_per_household'],
      dtype='str')

In [35]:
Y=df_encoded['median_house_value']#o/p dataset

#### horizontal splitting(train,test)

In [36]:
from sklearn.model_selection import train_test_split

In [37]:
# Split the data into training (80%) and testing (20%) sets
X_train_new, X_test_new, Y_train_new, Y_test_new = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

In [38]:
X_train_new

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN,rooms_per_household,bedrooms_per_room,population_per_household
14196,-117.03,32.71,33.0,3126.0,627.0,2300.0,623.0,3.2596,False,False,False,True,5.017657,0.200576,3.691814
8267,-118.16,33.77,49.0,3382.0,787.0,1314.0,756.0,3.8125,False,False,False,True,4.473545,0.232703,1.738095
17445,-120.48,34.66,4.0,1897.0,331.0,915.0,336.0,4.1563,False,False,False,True,5.645833,0.174486,2.723214
14265,-117.11,32.69,36.0,1421.0,367.0,1418.0,355.0,1.9425,False,False,False,True,4.002817,0.258269,3.994366
2271,-119.80,36.78,43.0,2382.0,431.0,874.0,380.0,3.5542,True,False,False,False,6.268421,0.180940,2.300000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,-117.96,33.78,35.0,1330.0,201.0,658.0,217.0,6.3700,False,False,False,False,6.129032,0.151128,3.032258
11964,-117.43,34.02,33.0,3084.0,570.0,1753.0,449.0,3.0500,True,False,False,False,6.868597,0.184825,3.904232
5390,-118.38,34.03,36.0,2101.0,569.0,1756.0,527.0,2.9344,False,False,False,False,3.986717,0.270823,3.332068
860,-121.96,37.58,15.0,3575.0,597.0,1777.0,559.0,5.7192,False,False,False,False,6.395349,0.166993,3.178891


#### standardization

In [39]:
from sklearn.preprocessing import StandardScaler

In [40]:
scaler_new = StandardScaler()

# Fit and transform the training features
X_train_scaled_new = scaler_new.fit_transform(X_train_new) ## mean and sd

# Transform the testing features using the training set scaler
X_test_scaled_new = scaler_new.transform(X_test_new) # mean and sd

### use regression model

In [41]:

from sklearn.linear_model import LinearRegression

In [42]:
lr = LinearRegression()

In [44]:
lr.fit(X_train_scaled_new,Y_train_new)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](15,)","[-56276.06,-56638.23, 14122.3 ,..., 7699.45, 16809.1 , 750.37]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,2.072e+05
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,15
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(15)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](15,)","[254.27,195.97,181.09,..., 25.96, 20.46, 13.5 ]"


In [45]:
X_test_new = X_test_new.fillna(X_train_new.median())
X_test_scaled_new = scaler_new.transform(X_test_new)
y_predict = lr.predict(X_test_scaled_new)
y_predict

array([ 36496.15659408, 137597.76591237, 293482.54728944, ...,
       447837.04647878, 117275.9214608 , 185597.46125194], shape=(4128,))

In [48]:
### model evaluation 
from sklearn.metrics import r2_score,root_mean_squared_error
print("RMSE:",root_mean_squared_error(Y_test_new,y_predict))
print("R2_Score:",r2_score(Y_test_new,y_predict))

RMSE: 69127.03829924983
R2_Score: 0.6353392335238193


>> this is not good model

## random forest regression

> ensemble learning algorithm

> model used = decision tree

> for classification - majority

> for regeression - average

<img src="https://www.slideteam.net/media/catalog/product/cache/1280x720/f/r/framework_of_random_forest_regression_algorithm_ppt_sample_slide01.jpg">

In [49]:
## Change the model to RandomForesetRegression 
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42) 
rf.fit(X_train_scaled_new,Y_train_new)
rfpredict = rf.predict(X_test_scaled_new)
print("RMSE:",root_mean_squared_error(Y_test_new,rfpredict))
print("R2_Score:",r2_score(Y_test_new,rfpredict))

RMSE: 49746.84462729571
R2_Score: 0.8111468563276775


>> Random Forest  Regressor has higher performance compare to Linear Regression

In [3]:
import pickle 

pickle.dump(rf,open("rfmodel.pkl","wb"))